# Detección de anomalías en control de calidad con EllipticEnvelope

## Caso a resolver

Una línea fabrica piezas mecanizadas y mide diámetro, peso, dureza, rugosidad y tiempo de ciclo. El área de calidad necesita priorizar piezas que se alejan del comportamiento normal para enviarlas a una segunda inspección.

Usaremos EllipticEnvelope de scikit-learn. No usaremos Isolation Forest, SVM ni Local Outlier Factor.

## Objetivos

- Crear datos sintéticos con unidades físicas plausibles.
- Explicar el problema y el fundamento estadístico.
- Entrenar sólo con piezas normales.
- Detectar combinaciones anómalas de mediciones.
- Interpretar métricas, scores y errores.
- Analizar sensibilidad y concluir con un plan productivo.

## 1. Contexto y método

La variación normal de una pieza no es cero. Un diámetro puede cambiar ligeramente, el peso puede acompañarlo y la rugosidad puede relacionarse con la dureza. Una pieza debe marcarse cuando su combinación de mediciones es poco compatible con el proceso estable.

EllipticEnvelope estima una región elíptica de normalidad usando centro y covarianza. Es apropiado como ejercicio cuando las mediciones normales tienen una distribución aproximadamente multivariada normal.

El modelo se entrenará con piezas normales. Las etiquetas de defecto se conservarán sólo para evaluar, no para ajustar el detector.

## 1.1 Principios estadísticos

EllipticEnvelope utiliza la distancia de Mahalanobis:

D cuadrada = (x - media) transpuesta por covarianza inversa por (x - media)

Esta distancia considera escala y correlación. Una desviación en peso puede ser normal si ocurre junto con una desviación coherente en diámetro. En cambio, una combinación improbable de diámetro, peso y rugosidad produce una distancia mayor.

La covarianza describe cómo se mueven juntas las mediciones. La envolvente estima una frontera y contamination define la proporción esperada de observaciones extremas.

Ventajas: interpretación estadística, consideración de correlaciones y buena explicación para control de proceso.

Limitaciones: supuesto elíptico, sensibilidad a cambios de máquina o producto y necesidad de segmentar si existen varios regímenes de operación.

## 2. Configuración

Importamos las herramientas de datos, visualización, escalamiento, EllipticEnvelope y métricas. La semilla hace reproducible el experimento en Colab.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import EllipticEnvelope
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
sns.set_theme(style='whitegrid')
print('Entorno listo. Semilla:', RANDOM_STATE)

### Explicación detallada del código

numpy genera las mediciones. pandas crea la tabla. StandardScaler equilibra las unidades. EllipticEnvelope aprende la frontera estadística. Las métricas comparan las alertas con los defectos sintéticos. La semilla permite reproducir las mismas piezas y resultados.

## 3. Generación de datos realistas

Se generan 1,200 piezas normales y 60 piezas anómalas. Las unidades representan un proceso de manufactura:

- diámetro en milímetros;
- peso en gramos;
- dureza en Rockwell C;
- rugosidad en micrómetros Ra;
- tiempo de ciclo en segundos.

La población normal incluye correlaciones de proceso. Los defectos simulan desviación dimensional, cambio de material, superficie rugosa y desgaste o problema de proceso.

In [ ]:
n_normales, n_anomalias = 1200, 60
medias = np.array([50.00, 250.0, 60.0, 1.80, 42.0])
desv = np.array([0.04, 3.0, 2.5, 0.35, 2.5])
correlaciones = np.array([
    [1.00, .55, .05, -.05, .20],
    [.55, 1.00, .10, -.05, .25],
    [.05, .10, 1.00, -.35, .10],
    [-.05, -.05, -.35, 1.00, .15],
    [.20, .25, .10, .15, 1.00]
])
covarianza = np.outer(desv, desv) * correlaciones
normales = rng.multivariate_normal(medias, covarianza, size=n_normales)
normales[:, 3] = np.clip(normales[:, 3], 0.5, 4.0)
normales[:, 4] = np.clip(normales[:, 4], 25, 65)
anomalias = rng.multivariate_normal(medias, covarianza, size=n_anomalias)
tipos = rng.choice(['dimension', 'material', 'superficie', 'proceso'], size=n_anomalias)
for i, tipo in enumerate(tipos):
    if tipo == 'dimension':
        anomalias[i, 0] += rng.choice([-1, 1]) * rng.uniform(.25, .55)
        anomalias[i, 1] += rng.choice([-1, 1]) * rng.uniform(12, 28)
    elif tipo == 'material':
        anomalias[i, 1] += rng.choice([-1, 1]) * rng.uniform(18, 35)
        anomalias[i, 2] += rng.choice([-1, 1]) * rng.uniform(8, 15)
    elif tipo == 'superficie':
        anomalias[i, 3] += rng.uniform(2.0, 5.0)
        anomalias[i, 2] += rng.choice([-1, 1]) * rng.uniform(5, 10)
    else:
        anomalias[i, 4] += rng.uniform(15, 35)
        anomalias[i, 3] += rng.uniform(.8, 2.5)
columnas = ['diametro_mm', 'peso_g', 'dureza_hrc', 'rugosidad_ra_um', 'tiempo_ciclo_s']
df = pd.concat([pd.DataFrame(normales, columns=columnas).assign(es_defecto_real=0),
                pd.DataFrame(anomalias, columns=columnas).assign(es_defecto_real=1)],
               ignore_index=True)
df.insert(0, 'pieza_id', [f'P-{i:05d}' for i in range(1, len(df)+1)])
df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
variables = columnas
print(f'Piezas: {len(df):,} | Defectos sintéticos: {df.es_defecto_real.sum():,}')
display(df.head())

### Explicación detallada de la generación

La matriz de correlaciones hace que los datos normales parezcan mediciones de un proceso, no cinco columnas independientes. La covarianza combina correlación y dispersión de cada instrumento.

Las anomalías no utilizan valores negativos abstractos. Se modifican mediciones físicas con magnitudes plausibles. La rugosidad, el peso, el diámetro y el ciclo son positivos; la dureza también queda dentro de valores razonables para una pieza metálica. La etiqueta sólo se utiliza después para evaluar.

## 4. Validación inicial

Antes del detector, verificamos faltantes, estadísticos y valores físicos. Esto separa problemas de calidad de datos de defectos reales del proceso.

In [ ]:
print('Valores faltantes:')
print(df[variables].isna().sum())
display(df[variables].describe().round(3))
print('Controles físicos:')
print({
    'diametros_no_positivos': int((df.diametro_mm <= 0).sum()),
    'pesos_no_positivos': int((df.peso_g <= 0).sum()),
    'rugosidades_no_positivas': int((df.rugosidad_ra_um <= 0).sum()),
    'ciclos_no_positivos': int((df.tiempo_ciclo_s <= 0).sum())
})

### Interpretación de la validación

Los controles deben devolver cero. Si aparecieran valores imposibles, se investigarían sensores, unidades o captura antes de entrenar. El margen de tolerancia de calidad no se decide sólo con el detector: debe definirse contra especificaciones de ingeniería.

## 4.1 Distribuciones de las mediciones

Los histogramas muestran objetivos, dispersión y colas por variable. La distribución se debe revisar por lote, turno, máquina y producto en una aplicación real.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
for ax, variable in zip(axes.ravel(), variables):
    sns.histplot(data=df, x=variable, hue='es_defecto_real', bins=35, kde=True,
                 palette={0:'#4C78A8', 1:'#E45756'}, alpha=.45, ax=ax)
    ax.set_title(variable)
axes.ravel()[-1].axis('off')
plt.tight_layout()
plt.show()

### Interpretación de distribuciones

Las piezas normales deben concentrarse alrededor de objetivos. Los defectos pueden aparecer en las colas, pero una cola no significa automáticamente rechazo: puede representar una condición permitida o un cambio de referencia. La visualización ayuda a decidir si el proceso parece compatible con una envolvente elíptica.

## 4.2 Relaciones entre variables

Las relaciones son importantes porque Mahalanobis utiliza covarianzas. Mostramos pares donde esperamos relaciones de proceso.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, (x, y) in zip(axes, [('diametro_mm','peso_g'), ('dureza_hrc','rugosidad_ra_um'), ('rugosidad_ra_um','tiempo_ciclo_s')]):
    sns.scatterplot(data=df, x=x, y=y, hue='es_defecto_real',
                    palette={0:'#4C78A8', 1:'#E45756'}, alpha=.65, ax=ax)
    ax.set_title(f'{x} vs {y}')
plt.tight_layout()
plt.show()

### Interpretación de relaciones

Diámetro y peso pueden aumentar juntos. Dureza y rugosidad pueden mostrar una relación inversa. Una pieza que rompe varias relaciones normales puede ser más anómala que otra que sólo tiene una medición alta.

## 4.3 Correlación y resumen por grupo

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df[variables].corr(), annot=True, fmt='.2f', cmap='vlag', center=0)
plt.title('Correlación entre mediciones')
plt.show()
display(df.groupby('es_defecto_real')[variables].agg(['median', 'mean', 'std', 'max']).round(3))

### Lectura del resumen

La tabla permite contrastar la población normal con la población defectuosa. Las diferencias orientan la investigación, pero no deben reemplazar las especificaciones técnicas. La correlación aporta información para la covarianza y para interpretar distancia de Mahalanobis.

## 5. Preparación y escalamiento

Estandarizamos porque las variables están en milímetros, gramos, dureza, micrómetros y segundos. Luego separamos las piezas normales para entrenar sólo con el proceso estable.

In [ ]:
X = df[variables].copy()
y_real = df['es_defecto_real'].to_numpy()
escalador = StandardScaler()
X_escalada = escalador.fit_transform(X)
X_entrenamiento = X_escalada[y_real == 0]
print('Matriz total:', X_escalada.shape)
print('Matriz normal de entrenamiento:', X_entrenamiento.shape)
print('Medias escaladas:', X_escalada.mean(axis=0).round(3))

### Explicación detallada

X excluye pieza_id y la etiqueta. El escalador evita que gramos dominen sobre milímetros. X_entrenamiento contiene únicamente piezas normales; así la media y covarianza representan el proceso de referencia y no los defectos inyectados.

## 6. Entrenamiento con EllipticEnvelope

Ajustamos la envolvente con las piezas normales y aplicamos la frontera a todas las piezas. La contaminación inicial es 5%, cercana a la proporción de defectos del benchmark.

In [ ]:
modelo = EllipticEnvelope(contamination=0.05, random_state=RANDOM_STATE)
modelo.fit(X_entrenamiento)
prediccion = modelo.predict(X_escalada)
df['prediccion_anomalia'] = (prediccion == -1).astype(int)
df['mahalanobis_score'] = -modelo.decision_function(X_escalada)
print(f'Alertas generadas: {df.prediccion_anomalia.sum():,} ({df.prediccion_anomalia.mean():.1%})')
display(df.sort_values('mahalanobis_score', ascending=False).head(10))

### Explicación detallada del entrenamiento

El modelo estima centro y covarianza robustos de la población normal. predict devuelve 1 dentro de la envolvente y -1 fuera. El score invertido ordena las piezas por alejamiento de la frontera. Un score alto significa prioridad de inspección, no probabilidad de defecto.

## 7. Evaluación del detector

In [ ]:
y_pred = df['prediccion_anomalia']
precision = precision_score(y_real, y_pred)
recall = recall_score(y_real, y_pred)
print(f'Precisión: {precision:.1%}')
print(f'Recall:    {recall:.1%}')
print(classification_report(y_real, y_pred, target_names=['normal', 'defecto'], digits=3))
cm = confusion_matrix(y_real, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Pred. normal', 'Pred. defecto'],
            yticklabels=['Real normal', 'Real defecto'])
plt.title('Matriz de confusión')
plt.xlabel('Predicción'); plt.ylabel('Real'); plt.show()

### Interpretación detallada de resultados

Un verdadero positivo es una pieza defectuosa enviada a inspección. Un falso positivo es una pieza normal que consume capacidad. Un falso negativo es un defecto que no se detecta y puede avanzar a ensamble o cliente.

En calidad, recall suele ser prioritario cuando el costo de liberar un defecto es alto. Después se controla precisión para evitar que la segunda inspección se sature. La exactitud global no debe ocultar defectos omitidos.

## 8. Interpretación de las alertas

Ordenamos las piezas por score y agregamos una hipótesis sencilla de causa para orientar al inspector. La hipótesis no sustituye análisis de causa raíz.

In [ ]:
alertas = df[df.prediccion_anomalia == 1].copy()
alertas['senal_calidad'] = np.select(
    [alertas.diametro_mm < 49.6, alertas.diametro_mm > 50.4,
     alertas.rugosidad_ra_um > 4.0, alertas.tiempo_ciclo_s > 60,
     alertas.peso_g > 280],
    ['diámetro bajo', 'diámetro alto', 'rugosidad elevada',
     'ciclo prolongado', 'peso elevado'],
    default='combinación multivariable')
display(alertas.sort_values('mahalanobis_score', ascending=False)
        [['pieza_id'] + variables + ['mahalanobis_score', 'senal_calidad']]
        .head(15).round(3))
display(alertas.senal_calidad.value_counts().rename_axis('señal').to_frame('alertas'))

### Interpretación operativa

El inspector debe revisar lote, máquina, turno, calibración, herramienta, materia prima y trazabilidad. Una señal como rugosidad elevada puede indicar desgaste, pero también un cambio autorizado de acabado. El score ayuda a priorizar y las mediciones ayudan a formular la investigación.

## 9. Sensibilidad a contamination

Probamos varias tasas esperadas de anomalías para mostrar cómo cambia el volumen de inspección y el equilibrio entre recall y precisión.

In [ ]:
resultados = []
for contamination in [.03, .05, .08, .10]:
    m = EllipticEnvelope(contamination=contamination, random_state=RANDOM_STATE)
    m.fit(X_entrenamiento)
    p = (m.predict(X_escalada) == -1).astype(int)
    resultados.append({'contaminacion': contamination, 'alertas': p.sum(),
                       'precision': precision_score(y_real, p),
                       'recall': recall_score(y_real, p)})
sensibilidad = pd.DataFrame(resultados)
display(sensibilidad.style.format({'contaminacion':'{:.0%}', 'precision':'{:.1%}', 'recall':'{:.1%}'}))
sensibilidad.plot(x='contaminacion', y=['precision','recall'], marker='o', ylim=(0,1),
                  figsize=(8,4), title='Sensibilidad del detector')
plt.ylabel('Métrica'); plt.show()

### Interpretación de sensibilidad

Con contaminación baja, se inspeccionan menos piezas, pero pueden aumentar falsos negativos. Con contaminación alta, se recuperan más defectos potenciales, pero aumenta la carga de inspección. El valor final debe considerar costo de scrap, retrabajo, paro de línea y escape al cliente.

## 10. Conclusiones

- EllipticEnvelope detecta combinaciones poco compatibles con un proceso normal usando centro y covarianza.
- La distancia de Mahalanobis considera correlaciones, no sólo diferencias individuales.
- Los datos tienen unidades físicas positivas y relaciones plausibles de manufactura.
- Entrenar con piezas normales aproxima una implementación de monitoreo de calidad.
- Recall es crítico cuando el costo de liberar un defecto supera el costo de inspección adicional.
- Las alertas requieren confirmación metrológica y análisis de causa raíz.
- En producción se debe segmentar por máquina, producto, lote y turno si sus distribuciones difieren.

### Plan recomendado

1. Usar históricos revisados y dividir por fecha.
2. Calibrar instrumentos y documentar especificaciones.
3. Revisar alertas por score y registrar la causa confirmada.
4. Medir estabilidad de alertas y falsos negativos.
5. Reentrenar cuando cambien producto, herramienta o condiciones de proceso.

## 11. Resumen reproducible

In [ ]:
print({'piezas': len(df), 'defectos_reales': int(y_real.sum()),
       'alertas': int(y_pred.sum()), 'precision': round(precision, 3),
       'recall': round(recall, 3)})